In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from shapely.geometry import Point
os.getcwd()

In [ ]:
def buffer_coastline(coastline, buffer, directory="./sites/poly_buffers", tabs=1):
    os.makedirs(directory, exist_ok=True)
    tab = '\t'
    print(f"{tab * tabs}Buffering coastline...")
    buffer_file = f"{directory}/coastline_{buffer}m.geojson"

    if os.path.exists(buffer_file):
        print(f"{tab * (tabs + 1)}Reading {buffer} m coast buffer...")
        buffer_poly = gpd.read_file(buffer_file).geometry
    else:
        print(f"{tab * (tabs + 1)}Creating {buffer} m coast buffer")
        buffer_poly = coastline.buffer(buffer)
        buffer_poly.to_file(buffer_file, driver='GeoJSON')
    
    return buffer_poly

In [ ]:
coastline_file = ".\\data\\coastline\\nz-coastlines-and-islands-polylines-topo-150k.gpkg"
coastpoly_file = ".\\data\\coastline\\nz-coastlines-and-islands-polygons-topo-150k.gpkg"
fault_file = 'C:\\Users\\jmc753\\Work\\NZ_CFM_v1_0\\Shapefiles\\NZ_CFM_v1_0.shp'
sites_dir = ".\\sites"
out_file = 'CUSP_v0-0-1_geoval'

backbone_res_km = 9   # National grid to use
coast_filters = [[2000, 1000]]   # [[Resolution, Buffer]]. Either in m, or if > 1km, must be 1km increments
fault_trace_filters = [3000, 1500, 0.99]   # [[Resolution, Buffer, SlipRate]]
fault_poly_buffers = [['CFM', 5000, 5000, 3000, 0.33, 3000]] #, ['sz', 5000, 5000, 3000, 0, 1000], ['py', 5000, 5000, 3000, 0, 1000]]  #  [Fault types (or specific fault name), hanging buffer, footbuffer, edgebuffer, min slip rate, site resolution]
fault_poly_buffers = []

premade_cropable_points = ['sz_py_polys_r1k_c1k', 'cfm_0-35mm_trace_r500m_b3k_c1k']  # Points that could be added to the new data, and can be cropped
# premade_cropable_points = []  # Points that could be added to the new data, and can be cropped
premade_final_points = ['validation_sites_KC_1m']  # Points that have been fully prepared and need to be included

clip_to_island = True # Clip all sites except highest resolution coast to only be onshore
final_coast_buffer = 0   # Final coast buffer - of all sites added previously, extract those within n m of the coast

min_area = 1e6  # sq m, min area for islands to be included

if backbone_res_km > 0:
    backbone = gpd.read_file(f"{sites_dir}\\national_{backbone_res_km}km_grid.geojson")
else:
    backbone = gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs='EPSG:2193')

coastline = gpd.read_file(coastline_file)
coastline = coastline[coastline['Area_SquareMeter'] > min_area]

coast_poly = gpd.read_file(coastpoly_file)
coast_poly = coast_poly[coast_poly['Area_SquareMeter'] > min_area]

faults = gpd.read_file(fault_file)
faults = gpd.sjoin(faults, coast_poly, how="inner", predicate="intersects")

clip_dir = f'{sites_dir}\\clipped_sites'
subset_dir = f'{sites_dir}\\N-S_sites'
poly_dir = f'{sites_dir}\\poly_buffers'
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(subset_dir, exist_ok=True)

coast_filters = coast_filters if len(coast_filters) > 0 else [[0, 0]]
fault_trace_filters = fault_trace_filters if len(fault_trace_filters) > 0 else [[]]
fault_poly_buffers = fault_poly_buffers if len(fault_poly_buffers) > 0 else [[]]

In [ ]:
points = gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs='EPSG:2193')
n_premade = np.zeros(len(premade_cropable_points), dtype=int)
if len(premade_cropable_points) > 0:
    print(f"Adding premade points...")
    for i, point_file in enumerate(premade_cropable_points):
        point_file = f"{sites_dir}\\{point_file}.geojson"
        if not os.path.exists(point_file):
            print(f"\t{point_file} does not exist, skipping...")
            continue
        print(f"\tReading from {point_file}...")
        new_points = gpd.read_file(point_file)
        n_premade[i] = new_points.shape[0]
        print(f"\t\tAdding {n_premade[i]} points...")
        points = pd.concat([points, new_points])
        print(f"\t\t{points.shape[0]} total points currently...")


print(f"\nAdding points from national {backbone_res_km} km backbone resolution...")
print(f"\tAdding {backbone.shape[0]} points...")
points = pd.concat([points, backbone[['geometry']]])
print(f"\t{points.shape[0]} total points currently...")


In [ ]:
print(f"Adding coastal points from {coastline.shape[0]} islands above {min_area * 1e-6:.0f} sqkm")
coast_filters = coast_filters if len(coast_filters) > 0 and isinstance(coast_filters[0], list) else [coast_filters]
coast_arr = np.array(coast_filters)
if coast_arr.shape[1] > 0:
    coast_arr = coast_arr[np.lexsort((coast_arr[:, 1], coast_arr[:, 0]))]
n_coast_points = np.zeros(coast_arr.shape[0], dtype=np.int32)

for ix, (res, buffer) in enumerate(coast_arr[1:], 1):
    res_units = 'm' if res < 1000 else 'km'
    res = int(res) if res < 1000 else int(res / 1000)
    buffer_units = 'm' if buffer < 1000 else 'km'
    print(f"\t{res} {res_units} spacing to {buffer if buffer < 1000 else buffer / 1000:.2f} {buffer_units} from coast")
    buffed_file = f'{clip_dir}\\coast_{min_area * 1e-6:.0f}_sqkm_{buffer}m_buff_{res}_{res_units}_res.geojson'
    if os.path.exists(buffed_file):
        print(f"\t\tReading pre-prepared {buffed_file}...")
        new_points = gpd.read_file(buffed_file)
    else:
        print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
        grid = gpd.read_file(f"{sites_dir}\\national_{res}{res_units}_grid.geojson")
        buffer_poly = buffer_coastline(coastline, buffer, tabs=2)
        print("\t\tFinding points in buffer...")
        intersect = grid.intersects(buffer_poly.unary_union)
        new_points = grid[intersect].reset_index()[['geometry']]
        new_points.to_file(buffed_file)

    n_coast_points[ix] = new_points.shape[0]
    print(f"\t\tAdding {n_coast_points[ix]} points...")
    points = pd.concat([points, new_points])
    print(f"\t\t{points.shape[0]} total points currently...")

In [ ]:
print(f"Adding fault_adjacent points....")
fault_trace_filters = fault_trace_filters if len(fault_trace_filters) > 0 and isinstance(fault_trace_filters[0], list) else [fault_trace_filters]
n_fault_points = np.zeros(len(fault_trace_filters), dtype=np.int32)

if len(fault_trace_filters[0]) > 0:
    for ix, (res, buffer, min_slip_rate) in enumerate(np.array(fault_trace_filters)):
        res_units = 'm' if res < 1000 else 'km'
        res = int(res) if res < 1000 else int(res / 1000)
        buffer_units = 'm' if buffer < 1000 else 'km'
        print(f"\t{res} {res_units} spacing to {buffer if buffer < 1000 else buffer / 1000:.2f} {buffer_units} from > {min_slip_rate} mm/yr faults")
        buffed_file = f'{clip_dir}\\faults_{str(min_slip_rate).replace(".", "-")}_mm_{buffer}m_buff_{res}_{res_units}_res.geojson'
        if os.path.exists(buffed_file):
            print(f"\t\tReading pre-prepared {buffed_file}...")
            new_points = gpd.read_file(buffed_file)
        else:
            print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
            grid = gpd.read_file(f"{sites_dir}\\national_{res}{res_units}_grid.geojson")
            print("\t\tBuffering faults...")
            buffer_poly = faults[faults.SR_pref > min_slip_rate].buffer(buffer)
            print("\t\tFinding points in buffer...")
            intersect = grid.intersects(buffer_poly.unary_union)
            new_points = grid[intersect].reset_index()[['geometry']]
            new_points.to_file(buffed_file)

        n_fault_points[ix] = new_points.shape[0]
        print(f"\t\tAdding {n_fault_points[ix]} points...")
        points = pd.concat([points, new_points])
        print(f"\t\t{points.shape[0]} total points currently...")

In [ ]:
print(f"Adding Fault Polygon Buffers")
n_poly_points = np.zeros(len(fault_poly_buffers), dtype=np.int32)
if len(fault_poly_buffers[0]) > 0:
    for ix, (fault_type, hangingbuff, footbuff, edgebuff, min_slip, res) in enumerate(fault_poly_buffers):
        fault_type = fault_type.replace(' ', '-')
        res_units = 'm' if res < 1000 else 'km'
        res = int(res) if res < 1000 else int(res / 1000)
        if fault_type == 'CFM':
            print(f"\t{res} {res_units} in > {min_slip} mm/yr {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers")
            poly_buffer_file = f"{fault_type}_hang-{hangingbuff / 1000:.0f}km_foot-{footbuff / 1000:.0f}km_edge-{edgebuff / 1000:.0f}km"
        else:
            print(f"\t{res} {res_units} in {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers")
            poly_buffer_file = f"{fault_type}_hang-{hangingbuff / 1000:.0f}km_foot-{footbuff / 1000:.0f}km_edge-{edgebuff / 1000:.0f}km"

        if fault_type == 'CFM':
            buffer_file = f"{clip_dir}\\{res}{res_units}_{poly_buffer_file}_{str(float(min_slip)).replace('.', '-')}mmyr.geojson"
        else:
            buffer_file = f"{clip_dir}\\{res}{res_units}_{poly_buffer_file}.geojson"

        if not os.path.exists(buffer_file):
            if os.path.exists(f"{poly_dir}\\{poly_buffer_file}.geojson"):
                buffer_poly = gpd.read_file(f"{poly_dir}\\{poly_buffer_file}.geojson")
                if fault_type == 'CFM':
                    n_poly = buffer_poly.shape[0]
                    buffer_poly = buffer_poly[buffer_poly['SlipRate'] > min_slip]
                    print(f"\t\t{buffer_poly.shape[0]}/{n_poly} fault segments above {min_slip:.02f} mm/yr...")
            else:
                print(f"\t{poly_dir}\\{poly_buffer_file}.geojson not found. Run generate_polygon_buffer.ipynb")
                continue
            print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
            grid = gpd.read_file(f"{sites_dir}\\national_{res}{res_units}_grid.geojson")
            print("\t\tFinding points in buffer...")
            intersect = grid.intersects(buffer_poly.unary_union)
            new_points = grid[intersect].reset_index()[['geometry']]
            print(f"\t\tWriting {buffer_file}...")
            new_points.to_file(buffer_file, driver='GeoJSON')
        else:
            print(f"\t\tReading {buffer_file}...")
            new_points = gpd.read_file(buffer_file)
        n_poly_points[ix] = new_points.shape[0]
        print(f"\t\tAdding {n_poly_points[ix]} points...")
        points = pd.concat([points, new_points])
        print(f"\t\t{points.shape[0]} total points currently...")


In [ ]:
if clip_to_island:
    pre_island_clip = points.shape[0]
    print('Clipping to Island Poly')
    points = gpd.sjoin(points, coast_poly, predicate="within")
    post_island_clip = points.shape[0]
    print(f'\t{post_island_clip} points...')

if coast_arr.shape[1] > 0 and coast_arr[0, 1] > 0:
    print(f"Adding highest res coast buffer (allows some offshore)")
    res, buffer = coast_arr[0]
    res_units = 'm' if res < 1000 else 'km'
    res = res if res < 1000 else int(res / 1000)
    buffer_units = 'm' if buffer < 1000 else 'km'
    print(f"\t{res} {res_units} spacing to {buffer if buffer < 1000 else int(buffer / 1000)} {buffer_units} from coast")
    buffed_file = f'{clip_dir}\\coast_{min_area * 1e-6:.0f}_sqkm_{buffer if buffer < 1000 else int(buffer / 1000)}_{buffer_units}_buff_{res}_{res_units}_res.geojson'
    if os.path.exists(buffed_file):
        print(f"\t\tReading pre-prepared {buffed_file}...")
        new_points = gpd.read_file(buffed_file)
    else:
        print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
        grid = gpd.read_file(f"{sites_dir}\\national_{res}{res_units}_grid.geojson")
        buffer_poly = buffer_coastline(coastline, buffer, tabs=2)
        print("\t\tFinding points in buffer...")
        intersect = grid.intersects(buffer_poly.unary_union)
        new_points = grid[intersect].reset_index()[['geometry']]
        new_points.to_file(buffed_file)

    n_coast_points[0] = new_points.shape[0]
    print(f"\t\tAdding {n_coast_points[0]} points...")
    points = pd.concat([points, new_points])
    print(f"\t\t{points.shape[0]} total points currently...")

In [ ]:
print(f'Check {points.shape[0]} points for duplicates before crop...')
points = points.drop_duplicates(ignore_index=True)
print(f'\t{points.shape[0]} points')
if final_coast_buffer > 0:
    previous_points = points.shape[0]
    print(f"Running {final_coast_buffer / 1000:.2f} km Final Coast Crop")
    buffer_poly = buffer_coastline(coastline, final_coast_buffer, tabs=1)
    print("\tFinding points in buffer...")
    intersect = points.intersects(buffer_poly.unary_union)
    points = points[intersect].reset_index()[['geometry']]
    print(f"\t\t{points.shape[0]} total points currently...")


In [ ]:
n_prepped = np.zeros(len(premade_final_points), dtype=int)
if len(premade_final_points) > 0:
    print(f"Adding fully prepared premade points...")
    for i, point_file in enumerate(premade_final_points):
        point_file = f"{sites_dir}\\{point_file}.geojson"
        if not os.path.exists(point_file):
            print(f"\t{point_file} does not exist, skipping...")
            continue
        print(f"\tReading from {point_file}...")
        new_points = gpd.read_file(point_file)
        n_prepped[i] = new_points.shape[0]
        print(f"\t\tAdding {n_prepped[i]} points...")
        points = pd.concat([points, new_points])
        print(f"\t\t{points.shape[0]} total points currently...")

In [ ]:
print("Drop Pacific Islands...")
points = points[(points.geometry.x < 2100000) & (points.geometry.y > 4740000)]
print(f'\t{points.shape[0]} points')

print('Formatting....')
points = points.reset_index(drop=True)
if 'siteId' not in points.columns:
    points['siteId'] = ''
points['Lon'] = np.round(points.geometry.x, 1)
points['Lat'] = np.round(points.geometry.y, 1)
points['Height'] = 0

points['siteId'] = [f"{round(points.loc[ix, 'Lon'])}_{round(points.loc[ix, 'Lat'])}" if not isinstance(points.loc[ix, 'siteId'], str) else points.loc[ix, 'siteId'] for ix in points.index.values]  # Set siteId to be based on NZTM location if not present

points = points[['siteId', 'Lon', 'Lat', 'Height', 'geometry']]

print(f'\t{points.shape[0]} points')
print('Remove duplicate points...')
points = points.drop_duplicates(ignore_index=True)
print(f'\t{points.shape[0]} points')

In [ ]:
print(f'Writing {points.shape[0]} sites to {out_file}.geojson...')
points.to_file(f"{sites_dir}\\{out_file}.geojson", driver='GeoJSON')

print("Splitting outputs into Hikurangi and Puysegur sections...")
wellington = Point([1749150, 5428092]) # Wellington coordinates in NZTM
te_anau = Point([1186710, 4957633])  # Te Anau coordinates in NZTM
distance = 350  # Distance South of Wellington in km to include for hikurangi

# For Hikurangi, find all centroids north of 350km south of Wellington
northern_section = points[(points.geometry.y > wellington.y) | (points.distance(wellington) < distance * 1e3)]
if northern_section.shape[0] > 0:
    northern_section.to_file(f"{subset_dir}\\{out_file}N.geojson", driver='GeoJSON')
    print(f"\t{northern_section.shape[0]} Northern sites in {out_file.replace('..geojson', 'N.geojson')}")

# For Puysegur, find all centroids within 350km of Te Anau
southern_section = points[(points.distance(te_anau) < distance * 1e3)]
if southern_section.shape[0] > 0:
    southern_section.to_file(f"{subset_dir}\\{out_file}S.geojson", driver='GeoJSON')
    print(f"\t{southern_section.shape[0]} Southern sites in {out_file.replace('..geojson', 'S.geojson')}")


In [ ]:
print(f"Writing metadata for {os.path.basename(out_file).split('.')[0]}")
with open(f"{sites_dir}\\{out_file}_meta.txt", 'w') as f:
    f.write(f"Name: {os.path.basename(out_file)}\n\n")
    f.write(f"National Resolution: {backbone_res_km} km ({backbone.shape[0]})\n")
    f.write(f"\nCoastal Strips:\n")
    if coast_arr.shape[1] > 1:
        for ix, (res, buffer) in enumerate(coast_arr[1:], 1):
            res_units = 'm' if res < 1000 else 'km'
            res = res if res < 1000 else int(res / 1000)
            buffer_units = 'm' if buffer < 1000 else 'km'
            offshore = ", inc. offshore" if ix == 0 else ""
            f.write(f"\t{ix}: {res} {res_units} spacing in {buffer if buffer < 1000 else buffer / 1000:.2f} {buffer_units} coast buffer ({n_coast_points[ix]}{offshore})\n")
    f.write(f"\nFault Traces:\n")
    if len(fault_trace_filters[0]) > 0:
        for ix, (res, buffer, min_slip_rate) in enumerate(fault_trace_filters):
            res_units = 'm' if res < 1000 else 'km'
            res = res if res < 1000 else int(res / 1000)
            buffer_units = 'm' if buffer < 1000 else 'km'
            f.write(f"\t{ix}: {res} {res_units} spacing for {buffer if buffer < 1000 else buffer / 1000:.2f} {buffer_units} around >{min_slip_rate} mm/yr faults ({n_fault_points[ix]})\n")
    f.write(f"\nFault Polygons:\n")
    if len(fault_poly_buffers[0]) > 0:
        for ix, (fault_type, hangingbuff, footbuff, edgebuff, min_slip, res) in enumerate(fault_poly_buffers):
            res_units = 'm' if res < 1000 else 'km'
            res = res if res < 1000 else int(res / 1000)
            if fault_type == 'CFM':
                f.write(f"\t{ix}: {res} {res_units} in > {str(min_slip).replace('.', '-')} mm/yr {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers ({n_poly_points[ix]})\n")
            else:
                f.write(f"\t{ix}: {res} {res_units} in {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers ({n_poly_points[ix]})\n")

    if len(premade_cropable_points) > 0:
        f.write(f"\nPremade Cropable Points:\n")
        for ix, point_file in enumerate(premade_cropable_points):
            f.write(f"\t{ix}: {point_file} ({n_premade[ix]})\n")

    if clip_to_island:
        f.write(f"\nIsland Clip: {pre_island_clip} -> {post_island_clip} sites\n")
    
    if coast_arr.shape[1] > 0 and coast_arr[0, 1] > 0:
        f.write(f"\nFinal Coastal Strip:\n")
        res, buffer = coast_arr[0]
        res_units = 'm' if res < 1000 else 'km'
        res = res if res < 1000 else int(res / 1000)
        buffer_units = 'm' if buffer < 1000 else 'km'
        f.write(f"\t0: {res} {res_units} spacing in {buffer if buffer < 1000 else int(buffer / 1000)} {buffer_units} coast buffer ({n_coast_points[0]}, inc. offshore)\n")

    if final_coast_buffer > 0:
        f.write(f"\nFinal Coast Crop: {final_coast_buffer} m - {previous_points} sites before cropping\n")

    if len(premade_final_points) > 0:
        f.write(f"\nPremade Final Points:\n")
        for ix, point_file in enumerate(premade_final_points):
            f.write(f"\t{ix}: {point_file} ({n_prepped[ix]})\n")

    f.write(f"\nTotal sites: {points.shape[0]}\n")
    f.write(f"Northern sites: {northern_section.shape[0]}\n")
    f.write(f"Southern sites: {southern_section.shape[0]}\n")
    
    
    for ix, meta_file in enumerate(premade_cropable_points):
        f.write(f"\n\n{ix}) Premade Cropabale Meta File ")
        if os.path.exists(f"{sites_dir}\\{meta_file}_meta.txt"):
            with open(f"{sites_dir}\\{meta_file}_meta.txt", 'r') as m:
                f.write(''.join(m.readlines()).replace('\n', '\n\t'))
        else:
            f.write(f"{sites_dir}\\{meta_file}_meta.txt not found")
    
    for ix, meta_file in enumerate(premade_final_points):
        f.write(f"\n\n{ix}) Premade Final Meta File ")
        if os.path.exists(f"{sites_dir}\\{meta_file}_meta.txt"):
            with open(f"{sites_dir}\\{meta_file}_meta.txt", 'r') as m:
                f.write(''.join(m.readlines()).replace('\n', '\n\t'))
        else:
            f.write(f"{sites_dir}\\{meta_file}_meta.txt Not found")